# Object Detection with YOLOv8 Fine-Tuning

This notebook fine-tunes a **YOLOv8** model on the [COCO 25-Class Object Detection](https://www.kaggle.com/datasets/malaychand/coco-25-class-object-detection-yolo-datasets) dataset.

## Overview

**YOLO (You Only Look Once)** is a single-stage object detection architecture that frames detection as a
regression problem, predicting bounding boxes and class probabilities directly from full images in a single
forward pass. YOLOv8 (by Ultralytics) improves on prior versions with an anchor-free detection head,
decoupled classification and regression branches, and a CSPDarknet backbone with C2f modules for
efficient multi-scale feature extraction.

**Plan:**
1. Install dependencies and explore the dataset
2. Create a YOLO-format data configuration
3. Fine-tune `yolov8n` (nano) for 25 epochs on a single GPU
4. Evaluate with mAP metrics and visualize predictions
5. Export the model to ONNX for deployment

---
## 1. Install Dependencies

In [ ]:
!pip install -q ultralytics

In [ ]:
# ============================================================
# Data Download (runtime fallback when dataset not mounted)
# ============================================================
import os, glob

KAGGLE_DATASET_PATH = "/kaggle/input/coco-25-class-object-detection-yolo-datasets"

if not os.path.exists(KAGGLE_DATASET_PATH) or not os.listdir(KAGGLE_DATASET_PATH):
    print("Dataset not mounted via /kaggle/input, downloading...")
    os.makedirs("/kaggle/working/data", exist_ok=True)
    os.system("kaggle datasets download -d malaychand/coco-25-class-object-detection-yolo-datasets -p /kaggle/working/data --unzip")
    candidates = glob.glob("/kaggle/working/data/**/detection", recursive=True)
    if candidates:
        KAGGLE_DATASET_PATH = str(os.path.dirname(candidates[0]))
    else:
        KAGGLE_DATASET_PATH = "/kaggle/working/data"
    print(f"Data downloaded. Root: {KAGGLE_DATASET_PATH}")
else:
    print(f"Dataset found at: {KAGGLE_DATASET_PATH}")


---
## 2. Imports & Configuration

In [ ]:
import os
import glob
import random
import shutil
import yaml
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from ultralytics import YOLO

# ── Paths ──────────────────────────────────────────────────────────────────────
# DATASET_ROOT set by download cell above
DATASET_ROOT = Path(KAGGLE_DATASET_PATH)
DETECTION_DIR = DATASET_ROOT / "detection"

# ── Debug: print actual directory structure so we can see what's there ──────────
print("=== Dataset root listing ===")
if DATASET_ROOT.exists():
    for p in sorted(DATASET_ROOT.iterdir()):
        print(f"  {p.name}/ " if p.is_dir() else f"  {p.name}")
        if p.is_dir():
            children = sorted(p.iterdir())
            for c in children[:10]:
                print(f"    {c.name}/ " if c.is_dir() else f"    {c.name}")
            if len(children) > 10:
                print(f"    ... ({len(children)} items total)")
else:
    print("  *** DATASET_ROOT does not exist! ***")

print()

# ── Source images and labels (flat, no train/val split) ────────────────────────
SRC_IMAGES = DETECTION_DIR / "images"
SRC_LABELS = DETECTION_DIR / "labels"

all_images = sorted(SRC_IMAGES.glob("*.jpg"))
print(f"Total source images: {len(all_images)}")
print(f"Total source labels: {len(list(SRC_LABELS.glob('*.txt')))}")

# ── Create train/val split (80/20) ────────────────────────────────────────────
WORK_DIR = Path("/kaggle/working/dataset")
TRAIN_IMAGES = WORK_DIR / "train" / "images"
TRAIN_LABELS = WORK_DIR / "train" / "labels"
VAL_IMAGES   = WORK_DIR / "val" / "images"
VAL_LABELS   = WORK_DIR / "val" / "labels"

for d in [TRAIN_IMAGES, TRAIN_LABELS, VAL_IMAGES, VAL_LABELS]:
    d.mkdir(parents=True, exist_ok=True)

# ── Training hyperparameters ────────────────────────────────────────────────────
EPOCHS   = 25
IMG_SIZE = 640
BATCH    = 16
DEVICE   = 0  # first GPU

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Shuffle and split
all_images_shuffled = all_images.copy()
random.shuffle(all_images_shuffled)
split_idx = int(len(all_images_shuffled) * 0.8)
train_imgs = all_images_shuffled[:split_idx]
val_imgs   = all_images_shuffled[split_idx:]

print(f"\nSplit: {len(train_imgs)} train, {len(val_imgs)} val")

# Symlink files into YOLO directory structure
def link_split(image_list, img_dst, lbl_dst):
    for img_path in image_list:
        lbl_path = SRC_LABELS / (img_path.stem + ".txt")
        dst_img = img_dst / img_path.name
        dst_lbl = lbl_dst / lbl_path.name
        if not dst_img.exists():
            os.symlink(img_path, dst_img)
        if lbl_path.exists() and not dst_lbl.exists():
            os.symlink(lbl_path, dst_lbl)

link_split(train_imgs, TRAIN_IMAGES, TRAIN_LABELS)
link_split(val_imgs, VAL_IMAGES, VAL_LABELS)

print(f"Train images dir: {len(list(TRAIN_IMAGES.glob('*')))} files")
print(f"Train labels dir: {len(list(TRAIN_LABELS.glob('*')))} files")
print(f"Val images dir:   {len(list(VAL_IMAGES.glob('*')))} files")
print(f"Val labels dir:   {len(list(VAL_LABELS.glob('*')))} files")

---
## 3. Explore the Dataset

In [ ]:
# ── Discover class names ───────────────────────────────────────────────────────
# This dataset has NO data.yaml, so we scan label files for unique class IDs
# and use a COCO-based fallback mapping for the 25 classes.

# Fallback: common COCO 25-class names (alphabetical order by class ID)
COCO_25_FALLBACK = [
    "person", "bicycle", "car", "motorcycle", "airplane",
    "bus", "train", "truck", "boat", "traffic light",
    "fire hydrant", "stop sign", "bird", "cat", "dog",
    "horse", "elephant", "bear", "zebra", "giraffe",
    "umbrella", "handbag", "bottle", "chair", "laptop",
]

# Scan ALL label files from the source directory to find unique class IDs
all_ids = set()
label_files = list(SRC_LABELS.glob("*.txt"))
print(f"Scanning {len(label_files)} label files for class IDs...")

for lbl in label_files:
    text = lbl.read_text().strip()
    if not text:
        continue
    for line in text.splitlines():
        parts = line.split()
        if parts:
            all_ids.add(int(parts[0]))

sorted_ids = sorted(all_ids)
num_ids = max(sorted_ids) + 1 if sorted_ids else 0
print(f"Found class IDs: {sorted_ids}")
print(f"Number of unique IDs: {len(sorted_ids)}, max ID: {max(sorted_ids) if sorted_ids else 'N/A'}")

# Use fallback names if the count matches, otherwise use generic names
if num_ids <= len(COCO_25_FALLBACK):
    CLASS_NAMES = COCO_25_FALLBACK[:num_ids]
else:
    CLASS_NAMES = [f"class_{i}" for i in range(num_ids)]

NUM_CLASSES = len(CLASS_NAMES)
print(f"\nNumber of classes: {NUM_CLASSES}")
print(f"Classes: {CLASS_NAMES}")

In [ ]:
def draw_yolo_boxes(image_path, label_path, class_names, ax):
    """Draw YOLO-format bounding boxes on an image."""
    img = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
    h, w, _ = img.shape
    ax.imshow(img)

    if label_path.exists():
        for line in label_path.read_text().strip().splitlines():
            parts = line.split()
            cls_id = int(parts[0])
            cx, cy, bw, bh = map(float, parts[1:5])

            # Convert YOLO normalised coords to pixel coords
            x1 = (cx - bw / 2) * w
            y1 = (cy - bh / 2) * h
            box_w = bw * w
            box_h = bh * h

            color = plt.cm.tab20(cls_id % 20)
            rect = patches.Rectangle(
                (x1, y1), box_w, box_h,
                linewidth=2, edgecolor=color, facecolor="none"
            )
            ax.add_patch(rect)
            label = class_names[cls_id] if cls_id < len(class_names) else str(cls_id)
            ax.text(
                x1, y1 - 4, label,
                fontsize=8, color="white",
                bbox=dict(facecolor=color, alpha=0.7, pad=1, edgecolor="none")
            )
    ax.axis("off")


# Show 8 random training samples
sample_images = sorted(TRAIN_IMAGES.glob("*"))
samples = random.sample(sample_images, min(8, len(sample_images)))

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, img_path in zip(axes.flat, samples):
    lbl_path = TRAIN_LABELS / (img_path.stem + ".txt")
    draw_yolo_boxes(img_path, lbl_path, CLASS_NAMES, ax)

fig.suptitle("Training Samples with Ground-Truth Boxes", fontsize=16)
plt.tight_layout()
plt.show()

---
## 4. Create YAML Config for YOLO Training

In [ ]:
# Build the data.yaml that Ultralytics expects
data_config = {
    "path": str(WORK_DIR),
    "train": "train/images",
    "val": "val/images",
    "nc": NUM_CLASSES,
    "names": CLASS_NAMES,
}

DATA_YAML = Path("/kaggle/working/data.yaml")
with open(DATA_YAML, "w") as f:
    yaml.dump(data_config, f, default_flow_style=False, sort_keys=False)

print(DATA_YAML.read_text())

---
## 5. Load Pretrained Model

In [ ]:
# Load YOLOv8 nano -- small and fast, ideal for Kaggle GPU quota
model = YOLO("yolov8n.pt")
print(model.info())

---
## 6. Train

In [ ]:
results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    project="runs/detect",
    name="yolov8n_coco25",
    seed=SEED,
    verbose=True,
)

---
## 7. Validate & Evaluate

In [ ]:
# Load the best checkpoint from training
# YOLO may nest the project path; find best.pt dynamically
import glob as _glob
_best_candidates = _glob.glob("**/yolov8n_coco25/weights/best.pt", recursive=True)
assert _best_candidates, "best.pt not found! Check training output above."
_best_path = _best_candidates[0]
print(f"Loading best model from: {_best_path}")
best_model = YOLO(_best_path)

# Run validation
metrics = best_model.val(data=str(DATA_YAML), device=DEVICE)

print(f"{'Metric':<25} {'Value':>10}")
print("-" * 37)
print(f"{'mAP@0.5':<25} {metrics.box.map50:>10.4f}")
print(f"{'mAP@0.5:0.95':<25} {metrics.box.map:>10.4f}")
print(f"{'Precision':<25} {metrics.box.mp:>10.4f}")
print(f"{'Recall':<25} {metrics.box.mr:>10.4f}")

In [ ]:
# Per-class mAP
per_class_map50 = metrics.box.maps  # array of mAP50 per class

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(range(NUM_CLASSES), per_class_map50, color="steelblue")
ax.set_yticks(range(NUM_CLASSES))
ax.set_yticklabels(CLASS_NAMES, fontsize=9)
ax.set_xlabel("mAP@0.5:0.95")
ax.set_title("Per-Class mAP@0.5:0.95")
ax.invert_yaxis()
for bar, val in zip(bars, per_class_map50):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center", fontsize=8)
plt.tight_layout()
plt.show()

---
## 8. Inference & Visualization

In [ ]:
# Run inference on validation images (no separate test split in this dataset)
val_image_list = sorted(VAL_IMAGES.glob("*.jpg"))
samples = random.sample(val_image_list, min(8, len(val_image_list)))

preds = best_model.predict(source=samples, imgsz=IMG_SIZE, device=DEVICE, verbose=False)

fig, axes = plt.subplots(2, 4, figsize=(22, 11))
for ax, result in zip(axes.flat, preds):
    img = cv2.cvtColor(result.orig_img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)

    if result.boxes is not None:
        for box in result.boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            conf = box.conf[0].cpu().item()
            cls_id = int(box.cls[0].cpu().item())
            label = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else str(cls_id)

            color = plt.cm.tab20(cls_id % 20)
            rect = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2, edgecolor=color, facecolor="none"
            )
            ax.add_patch(rect)
            ax.text(
                x1, y1 - 4, f"{label} {conf:.2f}",
                fontsize=8, color="white",
                bbox=dict(facecolor=color, alpha=0.7, pad=1, edgecolor="none")
            )
    ax.axis("off")

fig.suptitle("Predictions on Validation Images", fontsize=16)
plt.tight_layout()
plt.show()

---
## 9. Export to ONNX

In [ ]:
# Export the best checkpoint to ONNX for framework-agnostic deployment
onnx_path = best_model.export(format="onnx", imgsz=IMG_SIZE, simplify=True)
print(f"ONNX model exported to: {onnx_path}")
print(f"File size: {os.path.getsize(onnx_path) / 1e6:.1f} MB")

---
## Summary

- Fine-tuned **YOLOv8n** on the COCO 25-Class dataset for 25 epochs
- Evaluated with mAP@0.5 and mAP@0.5:0.95 metrics
- Visualized predictions with bounding boxes and confidence scores
- Exported the trained model to **ONNX** for deployment

### Next Steps
- Try larger variants (`yolov8s`, `yolov8m`) for higher accuracy
- Experiment with data augmentation (mosaic, mixup, HSV shifts)
- Add TensorRT export for optimized GPU inference